# Python. Работа с изображениями

## Мотивация

Датасет картинок — это не «просто картинки». Прежде чем модель что-то увидит, изображение надо **загрузить**, и именно на этом шаге чаще всего портятся данные.

Беда в том, что загрузка почти никогда не падает с ошибкой. JPEG откроется — но части деталей в нём уже нет. Палитровый GIF откроется — но в массиве окажутся номера цветов, а не яркость. Шестнадцатибитный снимок откроется — и молча превратится в два уровня серого. Маску сегментации уменьшат вместе с картинкой — и в разметке появятся классы, которых не было. Модель после этого учится, показывает метрики и делает вид, что всё в порядке.

Отдельная история — медицинские данные. В DICOM хранятся не яркости, а физическая величина: плотность ткани в единицах Хаунсфилда. Там же лежат геометрия среза и данные пациента. Если загрузить такой файл как обычную картинку, потеряется и шкала, и всё, что по ней можно посчитать.

Поэтому семинар — про загрузку: какие бывают форматы, что каждый из них хранит и теряет, и как превратить файл в массив, с которым можно работать дальше. Медицинские данные, GIF, изменение размера и гамма-коррекция разобраны во второй части ноутбука — она нужна для задач.

> **Как устроен семинар.** Ноутбук состоит из двух частей: **демонстрация**
> (разделы 1–6) — её показывают на занятии и повторяют за преподавателем, и
> **справочник** (разделы 7–10) — его не демонстрируют, он нужен при решении
> задач: там GIF, медицинские данные, изменение размера и гамма-коррекция.
>
> Свёрнутые блоки **🎙 Заметка преподавателя** — это то, что рассказывается
> вслух на занятии. Разворачивайте их потом, при подготовке к защите.
>
> **Демонстрацию** открывают из каталога семинара в репозитории
> (`seminars/15-python-images/`): `assets/` лежит рядом, и относительные пути
> в ячейках находят данные. Ячейки выполняются подряд сверху вниз — справочная
> часть пользуется переменными и функциями из демонстрационной. **Задачи**
> решают в домашнем каталоге (`~/seminar-15/`), как описано в `tasks.md`;
> `/tmp` не используем.

## 0. Данные и пакеты

В `assets/` лежат один и тот же синтетический снимок в разных форматах, анимация, шестнадцатибитный и многостраничный TIFF, маска сегментации и КТ-срез в DICOM.

```
%pip install pillow numpy tifffile pydicom matplotlib opencv-python imageio
```

Именно `%pip`, а не `!pip` и не `pip` в `%%bash`: магия ставит пакеты в то окружение, из которого запущено ядро ноутбука, а обычный `pip` в оболочке может указывать на другой интерпретатор — и следующий же `import` упадёт.

Все файлы синтетические и собраны скриптом `assets/make_assets.py` — им же их можно пересобрать или дополнить. Реальных снимков пациентов в репозитории нет: медицинские данные требуют согласия и обезличивания, и «взять картинку из интернета» здесь не работает.

In [ ]:
%%bash
ls -l assets

## 1. Изображение — это массив

`Image.open` возвращает объект Pillow: он знает формат, размер и режим, но **пикселей ещё не читал** — файл открывается лениво, декодирование происходит при первом обращении к данным.

- `format` — формат файла (`PNG`, `JPEG`, `GIF`, `TIFF`);
- `size` — `(ширина, высота)`;
- `mode` — что лежит в пикселе: `RGB`, `L` (серый), `P` (палитра), `RGBA`, `I;16` (16 бит).

In [ ]:
import numpy as np
from PIL import Image

In [ ]:
# Image.open читает только заголовок: пиксели ещё не декодированы.
image = Image.open("assets/photo.png")
print(image.format, image.size, image.mode)   # size — это (ширина, высота)

`np.asarray` превращает изображение в массив: `(высота, ширина, каналы)`. Смотреть на массив мы будем много раз, поэтому сразу заведём маленькую функцию.

In [ ]:
# Смотреть на массив будем много раз — заведём для этого одну функцию.
def describe(array):
    print(array.shape, array.dtype, array.min(), array.max())

In [ ]:
array = np.asarray(image)   # вот здесь пиксели наконец декодируются
describe(array)             # shape — это (высота, ширина, каналы): порядок обратный size

#### ❓ **Вопрос**: `image.size` печатает `(128, 128)`, и `array.shape` начинается с тех же чисел. Что сломается, если изображение не квадратное?

<details>

<summary><strong>Ответ</strong></summary>

Порядок осей у них разный: `size` — это `(ширина, высота)`, а `shape` — `(высота, ширина, каналы)`. На квадратном изображении ошибка не видна, а на кадре 640×480 `size` даст `(640, 480)`, `shape` — `(480, 640, 3)`. Отсюда берутся транспонированные картинки — изображение выглядит повёрнутым на 90° — и несовпадение размеров при ресайзе.

</details>

## 2. Форматы: что каждый хранит и что теряет

| Формат | Сжатие | Каналы и битность | Кадры | Где встречается |
|---|---|---|---|---|
| PNG | без потерь | серый, RGB, RGBA, палитра, 8 и 16 бит | один | маски, скриншоты, эталоны |
| JPEG | **с потерями** | серый, RGB, CMYK, только 8 бит | один | фотодатасеты, веб |
| GIF | без потерь | только палитра ≤ 256 цветов | много | анимации, старые данные |
| TIFF | обычно без потерь | любая битность, много каналов | много страниц | микроскопия, спутники, сканы |
| BMP | без сжатия | RGB, 8 бит | один | простые обмены, отладка |
| WebP | оба режима | RGB, RGBA | много | веб, компактное хранение |

У GIF в этой таблице оговорка: само сжатие действительно без потерь, но перевод фотографии в палитру из 256 цветов теряет цвета необратимо. «Без потерь» здесь про алгоритм сжатия, а не про то, что фотография доедет целой.

Разница между «без потерь» и «с потерями» — не про размер файла, а про то, совпадут ли пиксели после сохранения и чтения. JPEG раскладывает картинку по частотам и отбрасывает часть коэффициентов; вернуть их нельзя.

Один массив, два формата

Ниже — один и тот же массив, сохранённый в PNG и в JPEG.

In [ ]:
import os
# Один и тот же снимок в двух форматах — сравним размеры файлов в байтах.
print(os.path.getsize("assets/photo.png"), os.path.getsize("assets/photo.jpg"))

In [ ]:
# astype(int) обязателен: у uint8 вычитание уходит в переполнение, 10 - 20 даст 246.
jpeg = np.asarray(Image.open("assets/photo.jpg")).astype(int)
print("максимальное расхождение:", np.abs(array.astype(int) - jpeg).max())

#### ❓ **Вопрос**: JPEG получился в 19 раз меньше PNG при том же изображении. Можно ли хранить датасет в JPEG?

<details>

<summary><strong>Ответ</strong></summary>

Для фотографий, где нужен общий вид, — обычно да, так устроено большинство открытых датасетов. Нельзя там, где важно точное значение пикселя, а не «похожая картинка»: научные и измерительные данные, разметка, всё, что дальше сравнивают по числам. Здесь максимальное расхождение — 171 уровень из 255, то есть в отдельных точках цвет изменился радикально. Отдельная ловушка — пересохранение: каждый цикл «открыл — сохранил» добавляет новые потери.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Про JPEG полезно рассказать, что именно он выбрасывает. Картинка режется на блоки
8×8, каждый раскладывается по косинусным частотам, и высокие частоты — резкие
переходы — грубеют первыми. Отсюда знакомые «квадратики» вокруг текста и границ:
глаз к плавным переходам чувствительнее, чем к резким, на этом и построен формат.

Дальше следует неочевидное для многих: JPEG теряет данные при КАЖДОМ сохранении.
Открыли, чуть повернули, сохранили — потеряли ещё раз. В датасете, который
несколько раз пережали разные люди, артефактов может быть больше, чем полезного
сигнала, и модель прекрасно научится узнавать именно их.

</details>

### Обратный путь: из массива в файл

`Image.fromarray` превращает массив обратно в изображение, `save` записывает его, а формат берётся из расширения. У JPEG есть параметр `quality` (1–95): он и решает, сколько деталей выбросить.

In [ ]:
import pathlib

out = pathlib.Path.home() / "seminar-15"
out.mkdir(exist_ok=True)

In [ ]:
for quality in (10, 90):
    Image.fromarray(array).save(out / f"photo-q{quality}.jpg", quality=quality)
    print(quality, (out / f"photo-q{quality}.jpg").stat().st_size, "байт")

Байты мы сравнили, но занятие про незаметную порчу данных — посмотрим, что именно теряется. Картинки в ноутбуке рисует matplotlib; `imshow` для RGB достаточно одной строки.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, image, title in zip(axes, [array, Image.open(out / "photo-q90.jpg"), Image.open(out / "photo-q10.jpg")],
                            ["оригинал", "quality=90", "quality=10"]):
    ax.imshow(image)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

#### ❓ **Вопрос**: Файл сохранили с `quality=90`, открыли и сохранили ещё раз с тем же `quality=90`. Совпадут ли пиксели с первым файлом?

<details>

<summary><strong>Ответ</strong></summary>

Нет. Каждое сохранение в JPEG — это новое разложение по частотам и новое округление коэффициентов, поэтому потери накапливаются с каждым циклом «открыл — сохранил», даже если качество не меняли. Именно поэтому промежуточные результаты обработки держат в PNG, а в JPEG сохраняют один раз, в самом конце.

</details>

## 3. TIFF: 16 бит и несколько страниц

TIFF — контейнер: внутри может лежать что угодно, в том числе несколько страниц и данные точнее восьми бит на канал. Так хранят снимки микроскопа, спутниковые каналы и сканы.

Pillow с такими файлами работает неохотно, поэтому берут `tifffile`: он сразу отдаёт массив NumPy нужного типа.

In [ ]:
import tifffile
# tifffile сразу отдаёт массив нужного типа — Pillow с 16 битами работает неохотно.
scan = tifffile.imread("assets/scan16.tif")
describe(scan)   # uint16: 65 536 уровней вместо привычных 256

In [ ]:
# В одном TIFF может лежать несколько страниц — получаем трёхмерный массив.
stack = tifffile.imread("assets/stack.tif")
print("страниц:", stack.shape[0], "| размер страницы:", stack.shape[1:])

Шестнадцать бит — это 65 536 уровней вместо 256. Приводить их к `uint8` нужно осознанно: у Pillow `convert("L")` для режима `I;16` не растягивает диапазон, а обрезает его. Посчитаем, сколько разных значений остаётся.

In [ ]:
# Сколько различных значений осталось — этим и меряем потерю данных.
def levels(data):
    return len(np.unique(np.asarray(data)))

In [ ]:
print("в файле:", levels(scan))
# convert("L") для режима I;16 не растягивает диапазон, а обрезает его по 255.
print("после convert('L'):", levels(Image.open("assets/scan16.tif").convert("L")))

In [ ]:
# Честный путь: растянуть весь диапазон файла на 0…255 — уровни сохраняются.
scaled = (scan - scan.min()) / (scan.max() - scan.min())
print("после масштабирования:", levels((scaled * 255).round().astype(np.uint8)))

#### ❓ **Вопрос**: После `convert("L")` от снимка осталось два уровня яркости. Как правильно получить восьмибитную версию?

<details>

<summary><strong>Ответ</strong></summary>

Явным масштабированием: `(scan - scan.min()) / (scan.max() - scan.min()) * 255`, а для съёмки с известной шкалой — по фиксированным границам, а не по минимуму и максимуму конкретного кадра. Тогда понятно, какое значение во что перешло. Молчаливое `convert("L")` — типичный способ потерять данные ещё до обучения.

</details>

## 4. Нормализация

Модели работают с вещественными числами, поэтому `uint8` переводят в диапазон 0…1 или в нулевое среднее и единичную дисперсию.

Тип результата важен: `array / 255` даёт `float64` — вдвое больше памяти, чем `float32`, и в восемь раз больше исходного `uint8`. Стандарт для данных — `float32`.

In [ ]:
# Деление uint8 на число даёт float64 — вдвое больше памяти, чем нужно.
print((array / 255).dtype, (array / 255).nbytes // 1024, "КБ")

In [ ]:
# Приводим тип ДО деления: float32 — стандарт для данных, идущих в модель.
normalized = array.astype(np.float32) / 255
print(normalized.dtype, normalized.nbytes // 1024, "КБ", round(float(normalized.mean()), 4))

Вторая частая нормализация — вычесть среднее и поделить на разброс. Важная деталь: предобученные модели ждут её **поканально и с фиксированными числами датасета**, а не со средним по вашей картинке. Для моделей, обученных на ImageNet, это `mean = [0.485, 0.456, 0.406]` и `std = [0.229, 0.224, 0.225]` — именно поэтому такие константы публикуют в описании модели.

Считать среднее по каждому изображению отдельно нельзя: одна и та же кошка на светлом и тёмном снимке уехала бы в разные точки пространства признаков, и обучение пришлось бы вести на плавающей шкале.

In [ ]:
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)   # константы ImageNet
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
standardized = (normalized - mean) / std                   # вычитание идёт по каналам
print(standardized.dtype, standardized.shape, [round(float(v), 2) for v in standardized.mean(axis=(0, 1))])

## 5. Кроп

Кроп — это обычный срез массива, копирования не происходит:

`array[y1:y2, x1:x2]` — сначала строки, потом столбцы.

In [ ]:
# Кроп — обычный срез: сначала строки, потом столбцы.
crop = array[32:96, 32:96]
print(crop.shape, "| это вид, а не копия:", crop.base is not None)

В теме про изображения смотреть только на числа мало — выведем результат глазами.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(array)
axes[1].imshow(crop)
for ax in axes:
    ax.axis("off")

## 6. Что ещё ломается при загрузке

- **OpenCV читает в BGR.** `cv2.imread` возвращает каналы в обратном порядке; смешивать его с Pillow в одном коде — верный способ обучить модель на перепутанных цветах.
- **EXIF-поворот.** Фотографии с телефона хранят ориентацию отдельным тегом. Pillow сам его не применяет: нужен `ImageOps.exif_transpose`, иначе часть датасета окажется повёрнутой на 90°.
- **Альфа-канал и CMYK.** `RGBA` даёт четыре канала вместо трёх, а сканы из типографии бывают в `CMYK`. Приводите режим явно: `convert("RGB")`.
- **Ленивая загрузка.** `Image.open` читает только заголовок. Битый файл откроется, размер прочитается, а ошибка вылезет позже — чтобы поймать её сразу, нужен `image.load()`.
- **Слишком большие файлы.** Pillow ограничивает размер изображения (`DecompressionBombWarning`) — на спутниковых и микроскопических снимках лимит поднимают осознанно.

Ленивость проверяется в одну ячейку: обрежем файл до первых ста байт. Заголовок в них есть, пикселей нет.

In [ ]:
broken = out / "truncated.png"
broken.write_bytes(pathlib.Path("assets/photo.png").read_bytes()[:100])
print("размер прочитан:", Image.open(broken).size)   # заголовка хватило

In [ ]:
try:
    Image.open(broken).load()      # а вот пиксели декодировать уже не из чего
except OSError as error:
    print("а load() уже падает:", error)

In [ ]:
import cv2
# Один и тот же пиксель: OpenCV отдаёт BGR, Pillow — RGB. Каналы зеркальны.
print("cv2:", cv2.imread("assets/photo.png")[0, 0], "| Pillow:", array[0, 0])

#### ❓ **Вопрос**: Модель обучили на данных, загруженных через `cv2.imread`, а в продакшене картинки читает Pillow. Что произойдёт?

<details>

<summary><strong>Ответ</strong></summary>

Каналы поменяются местами: модель обучалась на BGR, а получает RGB. Ошибки не будет, точность просто упадёт — тем сильнее, чем важнее для задачи цвет. Лечится одной строкой (`cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`) и правилом использовать одну библиотеку загрузки на всём пути данных.

</details>

---

# Справочная часть

Дальше — то, что не показывают на занятии: пригодится при решении задач и при
подготовке к защите.

## 7. GIF: палитра и кадры

У GIF режим `P`: в файле лежит палитра до 256 цветов, а в пикселях — **номера** цветов в этой палитре. `np.asarray` вернёт именно номера, и обычная арифметика по ним бессмысленна.

In [ ]:
gif = Image.open("assets/photo.gif")
print(gif.mode, len(gif.getpalette()) // 3, "цветов в палитре")   # mode P — палитра
print("номера цветов:", np.asarray(gif)[0, :5])   # это индексы палитры, а не яркость

Ещё GIF бывает многокадровым. Pillow показывает первый кадр; остальные достаются через `seek`.

In [ ]:
animation = Image.open("assets/animation.gif")
print("кадров:", animation.n_frames)

In [ ]:
# Средняя яркость с округлением — понадобится и для кадров, и для гамма-коррекции.
def mean_brightness(image):
    return round(float(np.asarray(image).mean()), 2)

In [ ]:
animation.seek(3)   # перейти к четвёртому кадру: Pillow показывает по одному
# convert("L") превращает индексы палитры в яркость — усреднять индексы бессмысленно.
print("средняя яркость 4-го кадра:", mean_brightness(animation.convert("L")))

#### ❓ **Вопрос**: Что не так с выражением `np.asarray(Image.open("assets/photo.gif")).mean()`?

<details>

<summary><strong>Ответ</strong></summary>

Оно усредняет номера цветов в палитре, а не яркости. Номер 60 не в шестьдесят раз «ярче» номера 1 — палитра может быть упорядочена как угодно. Перед вычислениями палитровое изображение переводят в понятный режим: `convert("RGB")` или `convert("L")`.

</details>

## 8. Медицинские данные: DICOM

DICOM — формат и одновременно стандарт передачи медицинских изображений. Файл хранит не только пиксели, но и десятки тегов: модальность (CT, MR, US), геометрию среза, параметры съёмки и данные пациента.

Читает их `pydicom`; `ds.pixel_array` отдаёт массив NumPy.

In [ ]:
import pydicom
ds = pydicom.dcmread("assets/ct-slice.dcm")
# В DICOM кроме пикселей лежат теги: модальность, размер снимка, размер пикселя в мм.
print(ds.Modality, ds.Rows, ds.Columns, ds.PixelSpacing)

Главное отличие от обычной картинки: в пикселях лежит не яркость, а физическая величина в **единицах Хаунсфилда** (HU) — плотность ткани относительно воды. Чтобы её получить, сырое значение пересчитывают по тегам:

```
HU = pixel * RescaleSlope + RescaleIntercept
```

Шкала одинакова для всех аппаратов: воздух ≈ −1000, лёгкие ≈ −800…−600, жир ≈ −100, вода = 0, мягкие ткани ≈ 40, кость ≈ 300…1500. Именно поэтому её нельзя нормализовать по min/max отдельного среза: значение `40` должно означать мягкие ткани в любом файле датасета.

In [ ]:
# Эти четыре тега и превращают сырые числа в физическую величину и в картинку.
print(ds.RescaleSlope, ds.RescaleIntercept, ds.WindowCenter, ds.WindowWidth)

In [ ]:
# Перевод в единицы Хаунсфилда — обязательный первый шаг с любым КТ-снимком.
def to_hu(dataset):
    return dataset.pixel_array * dataset.RescaleSlope + dataset.RescaleIntercept

In [ ]:
hu = to_hu(ds)
describe(ds.pixel_array)      # сырые значения: беззнаковые, смысла сами по себе не имеют
print("в HU:", np.unique(hu))  # а это уже плотность тканей на общей для всех шкале

#### ❓ **Вопрос**: Почему КТ-срез нельзя нормализовать как обычную картинку, через min/max самого среза?

<details>

<summary><strong>Ответ</strong></summary>

Потому что HU — абсолютная шкала: 40 означает мягкие ткани в любом файле. Если делить на минимум и максимум конкретного среза, одна и та же ткань получит разные значения в разных файлах — достаточно, чтобы в кадр попал металлический имплант или воздух. Модель будет учиться на плавающей шкале. Вместо этого применяют окно с фиксированными границами.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Единицы Хаунсфилда названы по имени Годфри Хаунсфилда, который в 1971 году
собрал первый компьютерный томограф и в 1979-м получил за это Нобелевскую премию
по медицине. Шкала привязана к двум точкам: вода — ноль, воздух — минус тысяча.

Отсюда практическое следствие, которое стоит проговорить вслух: КТ — это
*измерительный прибор*, а не камера. Значение пикселя имеет физический смысл и
сравнимо между аппаратами и больницами. Именно поэтому нормализация по min/max
среза — это не «немного неточно», а потеря того, ради чего снимок делали.

</details>

### Окно: как из HU получают картинку

Диапазон HU шире, чем различает глаз и чем нужно модели, поэтому берут **окно** — интервал, который растягивается на всю яркость, а всё за его пределами обрезается. Окно задаётся центром и шириной, у распространённых задач они стандартные: мягкие ткани — 40/400, лёгкие — −600/1500, кость — 400/1800.

Хорошие файлы приносят подходящее окно с собой, в тегах `WindowCenter` и `WindowWidth`: здесь это 40 и 400, то есть границы −160 и 240 HU.

Как из единиц Хаунсфилда получают картинку

In [ ]:
# Окно: интервал HU, который растянут на всю яркость; остальное обрезано.
def apply_window(hu, center, width):
    low = center - width / 2
    high = center + width / 2
    return np.clip((hu - low) / (high - low), 0, 1)

In [ ]:
# Из диапазона 0…1 в байты — только для показа и сохранения в PNG.
def to_uint8(image):
    return (np.asarray(image) * 255).round().astype(np.uint8)

In [ ]:
# Окно берём из тегов файла — его туда положил тот, кто снимал.
windowed = apply_window(hu, float(ds.WindowCenter), float(ds.WindowWidth))
print("уровни в окне мягких тканей:", np.unique(to_uint8(windowed)))

Окно — это выбор, а не свойство файла: те же данные в лёгочном окне выглядят иначе, и различимыми становятся другие ткани.

In [ ]:
# Те же данные, другое окно — и различимыми становятся другие ткани.
lung = apply_window(hu, -600, 1500)
print("уровни в лёгочном окне:", np.unique(to_uint8(lung)))

Показывать такой массив нужно тоже аккуратно: серой палитрой и с явными границами, иначе `matplotlib` растянет контраст по данным и картинка окажется не той, что видит врач.

In [ ]:
# cmap="gray" — снимок не раскрашиваем; vmin/vmax — иначе matplotlib растянет контраст сам.
plt.imshow(windowed, cmap="gray", vmin=0, vmax=1)
plt.axis("off")

In [ ]:
# Персональные данные лежат внутри файла — переименовать его недостаточно.
print(ds.PatientName, ds.PatientID, ds.StudyDate, ds.BodyPartExamined)

#### ❓ **Вопрос**: В теге `PatientName` этого файла записано `PHANTOM^COURSE`. Что нужно сделать с настоящим снимком, прежде чем класть его в общий датасет?

<details>

<summary><strong>Ответ</strong></summary>

Обезличить: убрать или заменить теги с именем, идентификатором, датой рождения, номером карты и всем, что позволяет узнать пациента. Отдельно проверяют, что данные не «вшиты» в само изображение — на снимках УЗИ и рентгена подписи часто выжжены прямо в пикселях. Просто переименовать файл недостаточно: персональные данные лежат внутри.

</details>

Числа уровней мало что говорят — посмотрим оба окна рядом. Данные одни и те же, различимы в них разные ткани.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for ax, image, title in zip(axes, [windowed, lung], ["мягкие ткани 40/400", "лёгочное -600/1500"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

## 9. Изменение размера и маски сегментации

Изменение размера всегда что-то додумывает. Метод интерполяции выбирают по смыслу данных:

- `BILINEAR`, `BICUBIC`, `LANCZOS` — для изображений: усредняют соседние пиксели;
- (в Pillow эти константы живут в `Image.Resampling`; старая запись `Image.BILINEAR` пока работает, но в новом коде пишут полную)
- `NEAREST` — для масок сегментации: берёт значение ближайшего пикселя и не создаёт новых.

Маска хранит номера классов, а не яркость. Усреднение номеров классов даёт классы, которых нет в разметке.

Почему маску уменьшают только NEAREST

In [ ]:
mask = Image.open("assets/mask.png")
# 0, 128, 255 — это номера классов, а не оттенки серого.
print("классы в маске:", np.unique(np.asarray(mask)))

In [ ]:
print("после NEAREST :", levels(mask.resize((64, 64), Image.Resampling.NEAREST)))    # классы целы
print("после BILINEAR:", levels(mask.resize((64, 64), Image.Resampling.BILINEAR)))   # классов стало 53

#### ❓ **Вопрос**: В маске было три класса, после билинейного уменьшения стало 53 значения. Чем это обернётся при обучении?

<details>

<summary><strong>Ответ</strong></summary>

Значения между классами — это несуществующие классы. Функция потерь получит метки, которых нет в разметке, часть пикселей будет отнесена не туда, а границы объектов размоются. При этом ошибки не будет: обучение пойдёт, метрики просто окажутся хуже без видимой причины. Маски всегда уменьшают методом ближайшего соседа.

</details>

## 10. Гамма-коррекция

Значение пикселя в обычном изображении — не количество света, а закодированная величина: формат sRGB хранит яркость нелинейно, ближе к `яркость ** (1/2.2)`. Так устроено историческое наследие мониторов и заодно экономия бит на тёмных участках, которые глаз различает лучше.

Гамма-коррекция меняет это распределение:

```
out = 255 * (in / 255) ** gamma
```

При `gamma < 1` тёмные участки светлеют — так вытягивают детали в тенях. При `gamma > 1` картинка темнеет; это же преобразование применяют, чтобы перейти к линейной шкале перед физическими расчётами.

In [ ]:
# Нормируем в 0…1, возводим в степень, возвращаемся в байты; clip страхует от выхода за края.
def gamma_correction(image, gamma):
    return np.clip(255 * (image / 255) ** gamma, 0, 255).astype(np.uint8)

In [ ]:
# gamma < 1 вытягивает тени, gamma > 1 затемняет — смотрим по средней яркости.
print("исходное:", mean_brightness(array))
print("gamma=0.5:", mean_brightness(gamma_correction(array, 0.5)))
print("gamma=2.2:", mean_brightness(gamma_correction(array, 2.2)))

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, image, title in zip(axes, [array, gamma_correction(array, 0.5), gamma_correction(array, 2.2)],
                            ["исходное", "gamma=0.5", "gamma=2.2"]):
    ax.imshow(image)
    ax.set_title(title, fontsize=9)
    ax.axis("off")

#### ❓ **Вопрос**: Снимок вышел слишком тёмным, детали в тенях не разобрать. Какую гамму брать — больше или меньше единицы?

<details>

<summary><strong>Ответ</strong></summary>

Меньше единицы. Значения нормируются в 0…1, а возведение числа меньше единицы в степень меньше единицы его увеличивает — сильнее всего у маленьких значений, то есть в тенях. Средняя яркость здесь выросла со 106 до 146. Гамма больше единицы даёт обратный эффект и затемняет.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Про порядок каналов есть хорошая байка. OpenCV отдаёт BGR не по злому умыслу:
библиотеку начали писать в конце 1990-х, а камеры и форматы под Windows тогда
хранили пиксели именно в этом порядке. Менять поведение уже нельзя — сломается
весь написанный код, поэтому расхождение живёт четверть века.

Насколько это больно, зависит от того, важен ли задаче точный цвет: перепутанные
красный и синий меняют небо, кожу, дорожные знаки, а на медицинских изображениях
с псевдоцветом и на спутниковых каналах ломают всё. Ошибка при этом молчаливая:
ни исключения, ни предупреждения — только метрики почему-то хуже.

</details>

## Дополнительно

### Объёмные данные и серии срезов

Один файл DICOM — это один срез. Исследование целиком лежит серией файлов, которые сортируют по тегу `ImagePositionPatient` (или `InstanceNumber`) и собирают в трёхмерный массив. Расстояние между срезами берут из `SliceThickness` и `PixelSpacing` — воксель у КТ обычно не кубический, и об этом легко забыть при ресайзе.

Для нейровизуализации распространён другой формат — NIfTI (`.nii`, `.nii.gz`), его читает `nibabel`. Там объём и матрица преобразования в мировые координаты лежат в одном файле. Универсальный вариант для обоих форматов — `SimpleITK`.

### Единый интерфейс чтения

`imageio.v3.imread` открывает почти всё одним вызовом и сразу отдаёт массив:

```python
import imageio.v3 as iio

array = iio.imread("assets/photo.png")
frames = iio.imread("assets/animation.gif", index=None)
```

Это удобно для быстрых проверок. В рабочем коде чаще берут библиотеку под формат: Pillow для обычных картинок, `tifffile` для TIFF, `pydicom` для DICOM — так виден режим, битность и метаданные, а не только массив.

### Где брать изображения

Картинки из поиска — не датасет: у них неизвестна лицензия, происхождение и разметка. Берут опубликованные наборы с указанной лицензией и условиями использования. Для медицинских данных дополнительно нужны обезличивание и разрешение на использование — даже когда файлы уже лежат в открытом доступе.